# Bayesian Networks and Autoregressive Language Models

This notebook builds first- and second-order count-based language models using only
Python data structures and random sampling. It prints selected conditional probability
tables, verifies that every learned distribution sums to one, generates the required
text, and compares the two model orders.

In [1]:
from collections import Counter, defaultdict
from itertools import product
import random

SENTENCES = [
    "the cat sat on the mat",
    "the cat sat on the rug",
    "the dog sat on the mat",
    "the dog ran to the park",
    "the cat ran to the park",
    "the dog sat on the rug",
]
START, END = "<START>", "<END>"
TOKENISED = [sentence.lower().split() for sentence in SENTENCES]


class NGramLanguageModel:
    '''Maximum-likelihood n-gram model for P(next token | context).'''

    def __init__(self, order=1):
        if order < 1:
            raise ValueError("order must be at least 1")
        self.order = order
        self.counts = defaultdict(Counter)
        self.probabilities = {}
        self.vocabulary = set()

    def fit(self, tokenised_sentences):
        self.counts.clear()
        self.vocabulary = {START, END}
        for sentence in tokenised_sentences:
            tokens = [token.lower() for token in sentence]
            self.vocabulary.update(tokens)
            padded = [START] * self.order + tokens + [END]
            for index in range(self.order, len(padded)):
                context = tuple(padded[index - self.order:index])
                self.counts[context][padded[index]] += 1

        self.probabilities = {
            context: {
                token: count / sum(next_counts.values())
                for token, count in next_counts.items()
            }
            for context, next_counts in self.counts.items()
        }
        return self

    def _normalise_context(self, context):
        if self.order == 1 and isinstance(context, str):
            return (context,)
        context = tuple(context)
        if len(context) != self.order:
            raise ValueError(f"Expected a context of length {self.order}")
        return context

    def distribution(self, context):
        return dict(self.probabilities.get(self._normalise_context(context), {}))

    def most_probable_next(self, context):
        distribution = self.distribution(context)
        return max(distribution, key=distribution.get) if distribution else None

    def generate(self, mode="sample", max_tokens=20, rng=None):
        if mode not in {"sample", "greedy"}:
            raise ValueError("mode must be 'sample' or 'greedy'")
        rng = rng or random.Random()
        context = [START] * self.order
        generated = []

        for _ in range(max_tokens):
            distribution = self.distribution(context[-self.order:])
            if not distribution:  # defined fallback for an unseen context
                break
            if mode == "greedy":
                next_token = max(distribution, key=distribution.get)
            else:
                tokens, weights = zip(*distribution.items())
                next_token = rng.choices(tokens, weights=weights, k=1)[0]
            if next_token == END:
                break
            generated.append(next_token)
            context.append(next_token)
        return " ".join(generated)

    def normalisation_errors(self, tolerance=1e-12):
        return {
            context: sum(distribution.values())
            for context, distribution in self.probabilities.items()
            if abs(sum(distribution.values()) - 1.0) > tolerance
        }

    def nonzero_parameters(self):
        return sum(len(next_counts) for next_counts in self.counts.values())

    def unobserved_contexts(self):
        all_contexts = product(sorted(self.vocabulary), repeat=self.order)
        return sum(context not in self.counts for context in all_contexts)

## First-order conditional probability table and invariant tests

In [2]:
first_order = NGramLanguageModel(order=1).fit(TOKENISED)
selected_words = ["the", "cat", "dog", "sat", "ran"]

for word in selected_words:
    distribution = first_order.distribution(word)
    print(f"P(next | {word!r}) = {distribution}")
    prediction = first_order.most_probable_next(word)
    print(f"  argmax prediction: {prediction!r}")

errors = first_order.normalisation_errors()
assert not errors, errors
for context, distribution in first_order.probabilities.items():
    total = sum(distribution.values())
    assert abs(total - 1.0) < 1e-12
print(f"\nAll {len(first_order.probabilities)} first-order distributions sum to 1.0.")
print("Unobserved transitions have probability 0 because no smoothing is used.")

P(next | 'the') = {'cat': 0.25, 'mat': 0.16666666666666666, 'rug': 0.16666666666666666, 'dog': 0.25, 'park': 0.16666666666666666}
  argmax prediction: 'cat'
P(next | 'cat') = {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
  argmax prediction: 'sat'
P(next | 'dog') = {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
  argmax prediction: 'sat'
P(next | 'sat') = {'on': 1.0}
  argmax prediction: 'on'
P(next | 'ran') = {'to': 1.0}
  argmax prediction: 'to'

All 11 first-order distributions sum to 1.0.
Unobserved transitions have probability 0 because no smoothing is used.


## Generate and save at least 20 sampled sentences

In [3]:
rng = random.Random(7)
generated_twenty = [first_order.generate("sample", rng=rng) for _ in range(20)]
for index, sentence in enumerate(generated_twenty, start=1):
    print(f"{index:>2}. {sentence}")
assert len(generated_twenty) == 20

 1. the cat sat on the mat
 2. the cat sat on the rug
 3. the cat sat on the mat
 4. the park
 5. the cat sat on the rug
 6. the rug
 7. the cat ran to the dog sat on the dog sat on the park
 8. the park
 9. the dog sat on the dog ran to the mat
10. the rug
11. the park
12. the cat ran to the dog sat on the cat sat on the cat ran to the mat
13. the rug
14. the dog ran to the mat
15. the cat sat on the rug
16. the cat sat on the park
17. the dog ran to the dog ran to the mat
18. the cat sat on the mat
19. the cat sat on the park
20. the mat


## Greedy versus probabilistic generation

In [4]:
greedy_examples = [first_order.generate("greedy") for _ in range(5)]
sample_rng = random.Random(21)
sampled_examples = [first_order.generate("sample", rng=sample_rng) for _ in range(5)]

print("Greedy generation:")
for sentence in greedy_examples:
    print(" -", sentence)
print("\nSampling:")
for sentence in sampled_examples:
    print(" -", sentence)

print("\nUnique greedy outputs:", len(set(greedy_examples)))
print("Unique sampled outputs:", len(set(sampled_examples)))

Greedy generation:
 - the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on
 - the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on
 - the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on
 - the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on
 - the cat sat on the cat sat on the cat sat on the cat sat on the cat sat on

Sampling:
 - the dog sat on the dog ran to the cat sat on the cat ran to the cat ran to
 - the dog sat on the cat ran to the dog sat on the rug
 - the cat sat on the dog sat on the dog ran to the cat sat on the park
 - the mat
 - the park

Unique greedy outputs: 1
Unique sampled outputs: 5


## Second-order model: P(X_t | X_{t-2}, X_{t-1})

In [5]:
second_order = NGramLanguageModel(order=2).fit(TOKENISED)
assert not second_order.normalisation_errors()

selected_contexts = [
    (START, START),
    (START, "the"),
    ("the", "cat"),
    ("the", "dog"),
    ("sat", "on"),
]
for context in selected_contexts:
    distribution = second_order.distribution(context)
    print(f"P(next | {context}) = {distribution}")

second_rng = random.Random(7)
second_examples = [second_order.generate("sample", rng=second_rng) for _ in range(10)]
print("\nSecond-order samples:")
for sentence in second_examples:
    print(" -", sentence)

P(next | ('<START>', '<START>')) = {'the': 1.0}
P(next | ('<START>', 'the')) = {'cat': 0.5, 'dog': 0.5}
P(next | ('the', 'cat')) = {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
P(next | ('the', 'dog')) = {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
P(next | ('sat', 'on')) = {'the': 1.0}

Second-order samples:
 - the cat sat on the mat
 - the cat sat on the mat
 - the cat sat on the mat
 - the dog sat on the mat
 - the dog sat on the mat
 - the dog sat on the mat
 - the dog sat on the rug
 - the dog sat on the mat
 - the dog ran to the park
 - the dog sat on the mat


## Quantitative model comparison

In [6]:
def diversity(model, seed=123, samples=100):
    rng = random.Random(seed)
    generated = [model.generate("sample", rng=rng) for _ in range(samples)]
    return len(set(generated)), generated


rows = []
for model in (first_order, second_order):
    unique, examples = diversity(model)
    rows.append(
        (
            model.order,
            model.nonzero_parameters(),
            len(model.probabilities),
            model.unobserved_contexts(),
            unique,
        )
    )

print(
    f"{'Order':<7} {'Nonzero CPT entries':<21} {'Observed contexts':<19} "
    f"{'Unobserved contexts':<21} {'Unique / 100'}"
)
for row in rows:
    print(f"{row[0]:<7} {row[1]:<21} {row[2]:<19} {row[3]:<21} {row[4]}")

print("\nThe second-order model uses more context, but its context table is much sparser.")

Order   Nonzero CPT entries   Observed contexts   Unobserved contexts   Unique / 100
1       17                    11                  1                     40
2       19                    15                  129                   6

The second-order model uses more context, but its context table is much sparser.
